In [9]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from catboost import CatBoostRegressor
import xgboost as xgb
# 1. Chargement des données (Utilise tes noms de fichiers sauvegardés)
X_train = pd.read_csv("../features/global_features/X_train_umap.csv")
y_train = pd.read_csv("../features/global_features/y_train.csv")
X_test = pd.read_csv("../features/global_features/X_test_umap.csv")

In [10]:
# 2. Préparation
y_train = y_train.values.flatten()
test_ids = X_test['video_id']
X = X_train.drop(columns=['video_id'], errors='ignore')
X_test_final = X_test.drop(columns=['video_id'], errors='ignore')

# 3. Configuration du K-Fold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Listes pour stocker les scores de validation et les prédictions finales
oof_preds = np.zeros(len(X)) # Out-of-fold predictions
#test_preds = np.zeros(len(X_test_final))
cv_scores = []

X_train = pd.DataFrame(X_train)
y_train = pd.DataFrame(y_train)
X_test = pd.DataFrame(X_test)
# On isole l'ID pour la soumission finale (très important !)
test_ids = X_test['video_id'].copy()

# On définit les features en supprimant video_id
# errors='ignore' permet de ne pas planter si la colonne est déjà absente
X_train = X_train.drop(columns=['video_id'], errors='ignore')
X_test = X_test.drop(columns=['video_id'], errors='ignore')

# On s'assure que y_train est un array 1D pour les calculs de metrics
# y_train doit être la colonne 'score' uniquement
y_train_values = y_train.values.flatten()

In [11]:
# =========================
# 3) LightGBM params
# =========================
lgb_params2 = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.015,     # Plus lent pour ne pas rater l'optimum
    "num_leaves": 15,          # Très bas pour éviter l'overfitting
    "max_depth": 4,            # On force des arbres courts (plus robustes)
    "min_child_samples": 40,   # On force chaque feuille à avoir au moins 40 vidéos
    "feature_fraction": 0.5,   # On ne prend que 50% des colonnes par arbre
    "reg_alpha": 1.0,          # L1 plus fort
    "reg_lambda": 5.0,         # L2 plus fort
    "verbose": -1,
    "random_state": 42
}
lgb_params ={
    "objective": "regression",
    "metric": "rmse",
 'learning_rate': 0.024489296516914404, 
 'num_leaves': 98, 
 'max_depth': 9, 
 'min_child_samples': 51, 
 'feature_fraction': 0.6918923726742648, 
 'bagging_fraction': 0.8307449535209684, 
 'bagging_freq': 6, 
 'reg_alpha': 1.0725069061880557, 
 'reg_lambda': 0.0757098806219079,
 "random_state": 42}
# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds   = np.zeros(len(X_train))
test_preds  = np.zeros(len(X_test))
feature_cols = X_train.columns # Maintenant sans video_id
feature_imp = np.zeros(len(feature_cols))
print(feature_cols)
print(f"\n{'='*50}")
print(f"KFold CV — 5 folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_values)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_values[train_idx], y_train_values[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test) / 5
    feature_imp        += model.feature_importances_ / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

# Utilisation de y_train_values pour le calcul final
oof_rmse = np.sqrt(mean_squared_error(y_train_values, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")

Index(['umap_0', 'umap_1', 'umap_2', 'umap_3', 'umap_4', 'umap_5', 'umap_6',
       'umap_7', 'umap_8', 'umap_9', 'umap_10', 'umap_11', 'umap_12',
       'umap_13', 'umap_14', 'umap_15', 'umap_16', 'umap_17', 'umap_18',
       'umap_19'],
      dtype='object')

KFold CV — 5 folds
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6
[LightGBM] [Warning] feature_fraction is set=0.6918923726742648, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6918923726742648
[LightGBM] [Warning] bagging_fraction is set=0.8307449535209684, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8307449535209684
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6
[LightGBM] [Warning] feature_fraction is set=0.6918923726742648, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6918923726742648
[LightGBM] [Warning] bagging_fraction is set=0.8